# NB_MDP_20 · Silver (una entidad)

Lo llama `NB_MDP_00_orquestador` después del bronze de la misma entidad.

1. Lee de bronze **solo las filas de la corrida** (`_run_id`). Para reprocesar una corrida vieja, pasá su id en `bronze_run_id`.
2. Si la entidad tiene columnas en `meta.ENTITY_COLUMN`, selecciona, renombra, transforma y tipa. Si no tiene, pasa todas las columnas.
3. Aplica las reglas de `meta.DQ_RULE`: `WARN` cuenta, `DROP` manda la fila a `<esquema silver>.dq_rechazos` (sin columnas PII) y `FAIL` corta la entidad.
4. Deduplica por `PRIMARY_KEY`, quedándose con la versión más nueva (`ORDER_COLUMN`, o `_ingested_at`).
5. Escribe con `MERGE` (upsert por clave), `SCD2` (historia con vigencia), `OVERWRITE` o `APPEND`, según `SILVER_MODE`.
6. Si `DELETE_MODE` no es `NONE`, marca las bajas del origen sin borrar filas: `FLAG` (una columna del origen dice que la fila está anulada) o `ABSENT` (la clave no vino en una extracción completa, con un freno de `DELETE_MAX_PCT` %).

Con `MERGE` y `SCD2`, correrlo dos veces con los mismos datos da el mismo resultado.

**SCD2.** Cada clave tiene versiones con `_valid_from`, `_valid_to` y `_is_current`. Una versión nueva se abre solo si cambia alguna columna de `SCD2_COLUMNS` (sin valor: todas menos la clave y `ORDER_COLUMN`); las demás se actualizan en el lugar. La historia tiene la granularidad de las corridas: si una clave cambió dos veces entre corridas, queda el último estado.

In [ ]:
# Parámetros
run_id = "manual"
entity_json = "{}"
bronze_run_id = ""      # vacío = la corrida actual

In [ ]:
import json
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

e = json.loads(entity_json)
resultado = {"layer": "silver", "object_code": e.get("entity_code"), "status": None,
             "rows_read": None, "rows_written": None, "rows_rejected": None,
             "dq_warnings": None, "error": None}

AUDITORIA = ["_run_id", "_ingested_at"]


def tabla_rechazos(silver_table):
    """silver.clientes -> silver.dq_rechazos ; LH.silver.clientes -> LH.silver.dq_rechazos"""
    partes = silver_table.split(".")
    return ".".join(partes[:-1] + ["dq_rechazos"])


def aplicar_columnas(df, columnas):
    if not columnas:
        return df
    exprs = [F.expr(c.get("transform_expr") or f"`{c['source_column']}`")
               .cast(c["target_type"]).alias(c["target_column"]) for c in columnas]
    return df.select(*exprs, *AUDITORIA)


def aplicar_calidad(df, reglas, columnas_pii):
    """Devuelve (df_ok, df_rechazos o None, avisos). Una regla que da NULL cuenta como incumplida."""
    avisos, rechazos = {}, None
    for r in reglas or []:
        cumple = F.coalesce(F.expr(r["rule_expr"]), F.lit(False))
        if r["action"] == "FAIL":
            n = df.where(~cumple).count()
            if n:
                raise ValueError(f"Regla {r['rule_name']} (FAIL): {n} filas no cumplen {r['rule_expr']}")
        elif r["action"] == "WARN":
            n = df.where(~cumple).count()
            if n:
                avisos[r["rule_name"]] = n
        elif r["action"] == "DROP":
            malas = df.where(~cumple).withColumn("_regla", F.lit(r["rule_name"]))
            rechazos = malas if rechazos is None else rechazos.unionByName(malas)
            df = df.where(cumple)
    if rechazos is not None:
        # Las columnas PII no se copian a la tabla de rechazos
        visibles = [c for c in rechazos.columns if c not in columnas_pii and c not in ("_regla", *AUDITORIA)]
        rechazos = rechazos.select(
            F.lit(e["entity_code"]).alias("entity_code"),
            F.col("_regla").alias("rule_name"),
            F.col("_run_id").alias("run_id"),
            F.to_json(F.struct(*visibles)).alias("row_json"),
            F.current_timestamp().alias("rejected_at"))
    return df, rechazos, avisos


def deduplicar(df, pk, order_column):
    if not pk:
        return df
    orden = [F.col(order_column).desc_nulls_last()] if order_column else []
    w = Window.partitionBy(*pk).orderBy(*orden, F.col("_ingested_at").desc())
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")


def filas_escritas(tabla):
    m = DeltaTable.forName(spark, tabla).history(1).select("operationMetrics").first()[0] or {}
    if "numTargetRowsInserted" in m:   # MERGE: numOutputRows incluye filas reescritas sin cambios
        return int(m["numTargetRowsInserted"]) + int(m.get("numTargetRowsUpdated", 0))
    return int(m.get("numOutputRows", 0))


# --- Historia (SCD2) y bajas del origen ------------------------------------------
COLS_SCD2 = ["_hash", "_valid_from", "_valid_to", "_is_current", "_is_deleted"]
SISTEMA = set(AUDITORIA) | {"_silver_updated_at", "_deleted_at", *COLS_SCD2}
INICIO = F.lit("1900-01-01 00:00:00").cast("timestamp")   # primera versión de una clave: vale desde siempre
FIN = F.lit("9999-12-31 23:59:59").cast("timestamp")      # versión vigente: vale hasta "nunca"


def lista(texto):
    return [k.strip() for k in (texto or "").split(",") if k.strip()]


def cond_clave(pk, a, b):
    """Igualdad null-safe por clave entre dos DataFrames con alias."""
    c = None
    for k in pk:
        x = F.col(f"{a}.`{k}`").eqNullSafe(F.col(f"{b}.`{k}`"))
        c = x if c is None else c & x
    return c


def columnas_negocio(df):
    return [c for c in df.columns if c not in SISTEMA]


def columnas_seguidas(df, pk, scd2_columns, order_column):
    """Las que abren una versión nueva al cambiar. Sin SCD2_COLUMNS: todas menos la clave y ORDER_COLUMN."""
    negocio = columnas_negocio(df)
    if scd2_columns:
        faltan = [c for c in scd2_columns if c not in negocio]
        if faltan:
            raise ValueError(f"SCD2_COLUMNS nombra columnas que no están en silver: {faltan}")
        return list(scd2_columns)
    return [c for c in negocio if c not in pk and c != order_column]


def marcar_borrado(df, delete_mode, flag_expr):
    """Agrega _is_deleted. FLAG: la fila viene marcada como baja en el origen. NONE y ABSENT: llega viva."""
    if delete_mode == "FLAG":
        return df.withColumn("_is_deleted", F.coalesce(F.expr(flag_expr).cast("boolean"), F.lit(False)))
    return df.withColumn("_is_deleted", F.lit(False))


def claves_ausentes(vivas, presentes, pk):
    """Claves vivas en silver que no vinieron en la extracción completa de esta corrida."""
    return (vivas.alias("v").join(presentes.select(*[F.col(f"`{k}`") for k in pk]).distinct().alias("p"),
                                  cond_clave(pk, "v", "p"), "left_anti"))


def controlar_bajas(n_vivas, n_ausentes, max_pct):
    """Freno ante una extracción parcial: no marca de baja más de DELETE_MAX_PCT % de las claves vivas."""
    if n_vivas and n_ausentes * 100.0 / n_vivas > float(max_pct):
        raise ValueError(f"Bajas por ausencia: faltan {n_ausentes} de {n_vivas} claves vivas "
                         f"({n_ausentes * 100.0 / n_vivas:.1f} % > DELETE_MAX_PCT {max_pct}). "
                         "¿La extracción vino incompleta? No se marcó ninguna baja ni se escribió silver.")


def agregar_columnas_faltantes(destino, df):
    """ALTER TABLE ADD COLUMNS con las columnas de df que la tabla todavía no tiene."""
    existentes = set(spark.table(destino).columns)
    nuevas = [f"`{n}` {t}" for n, t in df.dtypes if n not in existentes and n != "_accion"]
    if nuevas:
        spark.sql(f"ALTER TABLE {destino} ADD COLUMNS ({', '.join(nuevas)})")


def con_hash(df, seguidas):
    """Hash de las columnas seguidas. Una columna que la tabla todavía no tiene cuenta como NULL."""
    cols = [F.col(f"`{c}`") if c in df.columns else F.lit(None).alias(c) for c in seguidas]
    return df.withColumn("_hash", F.sha2(F.to_json(F.struct(*cols)), 256))


def forma_version(inc, seguidas):
    """Las columnas que tiene una tabla silver en SCD2, para crearla o completarla."""
    return (con_hash(inc, seguidas).withColumn("_valid_from", INICIO).withColumn("_valid_to", FIN)
            .withColumn("_is_current", F.lit(True)))


def inicio_candidato(df, order_column):
    """Desde cuándo vale una versión: ORDER_COLUMN si es fecha u hora, si no _ingested_at."""
    if order_column and dict(df.dtypes).get(order_column) in ("timestamp", "timestamp_ntz", "date"):
        return F.coalesce(F.col(f"`{order_column}`").cast("timestamp"), F.col("_ingested_at"))
    return F.col("_ingested_at")


def convertir_a_scd2(destino, seguidas):
    """Una tabla silver que venía con MERGE pasa a SCD2: cada fila queda como su primera versión vigente."""
    df = spark.table(destino)
    if "_is_current" in df.columns:
        return False
    df = (con_hash(df, seguidas)
          .withColumn("_valid_from", INICIO).withColumn("_valid_to", FIN)
          .withColumn("_is_current", F.lit(True))
          .withColumn("_is_deleted", F.coalesce(F.col("_is_deleted"), F.lit(False)) if "_is_deleted" in df.columns
                      else F.lit(False)))
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(destino)
    return True


def preparar_scd2(inc, actual, existieron, pk, seguidas, order_column, ausentes=None, pii=()):
    """Arma las filas para un único MERGE de SCD tipo 2.

    inc: una fila por clave de la corrida, con _is_deleted. actual: versiones vigentes de silver.
    existieron: claves que alguna vez estuvieron en silver. ausentes: versiones vigentes cuya clave
    no vino (bajas por ausencia). Devuelve (staged, fuera_de_orden). staged trae _accion:
      CERRAR   cierra la versión vigente (_valid_to y, si es baja, _is_deleted)
      TIPO1    misma versión: actualiza en el lugar las columnas que no se siguen
      INSERTAR versión nueva
    Una versión que nace cerrada no guarda las columnas PII que no se siguen (solo la vigente las tiene).
    """
    inc = con_hash(inc, seguidas).withColumn("_desde", inicio_candidato(inc, order_column))
    ren = {k: f"_a{i}" for i, k in enumerate(pk)}
    a = actual.select(*[F.col(f"`{k}`").alias(ren[k]) for k in pk],
                      F.col("_hash").alias("_a_hash"), F.col("_valid_from").alias("_a_desde"),
                      F.col("_ingested_at").alias("_a_ing"))
    ex = existieron.select(*[F.col(f"`{k}`").alias(f"_e{i}") for i, k in enumerate(pk)]).distinct()
    c_a = [F.col(f"`{k}`").eqNullSafe(F.col(ren[k])) for k in pk]
    c_e = [F.col(f"`{k}`").eqNullSafe(F.col(f"_e{i}")) for i, k in enumerate(pk)]
    j = (inc.join(a, c_a, "left").join(ex.withColumn("_existio", F.lit(True)), c_e, "left")
            .withColumn("_vigente", F.col("_a_hash").isNotNull())
            .withColumn("_existio", F.coalesce("_existio", F.lit(False)))
            # una corrida más vieja que la versión vigente (reproceso) no reescribe la historia
            .withColumn("_fuera", F.coalesce(F.col("_vigente") & (F.col("_ingested_at") < F.col("_a_ing")), F.lit(False)))
            # la versión nueva nunca empieza antes que la que cierra: los intervalos no se pisan
            .withColumn("_nf", F.greatest("_desde", F.coalesce("_a_desde", "_desde"))))
    fuera = j.where("_fuera").count()
    j = j.where(~F.col("_fuera"))
    borrada, cambio = F.col("_is_deleted"), F.col("_hash") != F.col("_a_hash")
    cols = [c for c in inc.columns if c not in ("_desde", "_valid_from", "_valid_to", "_is_current")]

    borrar_pii = [c for c in pii if c not in seguidas and c not in pk]

    def filas(cond, accion, desde, hasta, vigente, baja):
        def valor(c):
            if not vigente and c in borrar_pii:
                return F.lit(None).cast(dict(j.dtypes)[c]).alias(c)
            return F.col(f"`{c}`")
        return j.where(cond).select(*[valor(c) for c in cols if c != "_is_deleted"],
                                    desde.alias("_valid_from"), hasta.alias("_valid_to"),
                                    F.lit(vigente).alias("_is_current"), baja.alias("_is_deleted"),
                                    F.lit(accion).alias("_accion"))

    vig, nueva_clave = F.col("_vigente"), ~F.col("_vigente")
    partes = [
        filas(vig & ~borrada & ~cambio, "TIPO1", F.col("_a_desde"), FIN, True, F.lit(False)),
        filas(vig & (borrada | cambio), "CERRAR", F.col("_a_desde"), F.col("_nf"), False, borrada),
        filas(vig & ~borrada & cambio, "INSERTAR", F.col("_nf"), FIN, True, F.lit(False)),
        filas(nueva_clave & ~borrada, "INSERTAR",
              F.when(F.col("_existio"), F.col("_desde")).otherwise(INICIO), FIN, True, F.lit(False)),
        # llega ya dada de baja y nunca estuvo: queda una versión cerrada, para que los hechos viejos la encuentren
        filas(nueva_clave & borrada & ~F.col("_existio"), "INSERTAR", INICIO, F.col("_desde"), False, F.lit(True)),
    ]
    staged = partes[0]
    for p in partes[1:]:
        staged = staged.unionByName(p)
    if ausentes is not None:
        cierre = ausentes.select(*[F.col(f"`{k}`") for k in pk],
                                 F.greatest(F.current_timestamp(), F.col("_valid_from")).alias("_valid_to"),
                                 F.lit(False).alias("_is_current"), F.lit(True).alias("_is_deleted"),
                                 F.current_timestamp().alias("_silver_updated_at"), F.lit("CERRAR").alias("_accion"))
        staged = staged.unionByName(cierre, allowMissingColumns=True)
    return staged, fuera


def escribir_scd2(destino, staged, pk, seguidas, pii=()):
    """Un MERGE: cierra versiones, actualiza columnas tipo 1 de la vigente e inserta versiones nuevas.
    Al cerrar una versión se vacían las columnas PII que no se siguen: la historia no guarda datos personales viejos."""
    tipo1 = [c for c in columnas_negocio(staged) if c not in pk and c not in seguidas and c != "_accion"]
    cierre = {"_valid_to": "s._valid_to", "_is_current": "false",
              "_is_deleted": "s._is_deleted", "_silver_updated_at": "s._silver_updated_at",
              **{f"`{c}`": "NULL" for c in pii if c in tipo1}}
    cond = "s._accion IN ('CERRAR', 'TIPO1') AND t._is_current = true AND " + \
           " AND ".join(f"t.`{k}` <=> s.`{k}`" for k in pk)
    m = (DeltaTable.forName(spark, destino).alias("t").merge(staged.alias("s"), cond)
         .whenMatchedUpdate("s._accion = 'CERRAR'", cierre))
    if tipo1:
        hubo_cambio = " OR ".join(f"NOT (t.`{c}` <=> s.`{c}`)" for c in tipo1)
        m = m.whenMatchedUpdate(f"s._accion = 'TIPO1' AND ({hubo_cambio})",
                                {**{f"`{c}`": f"s.`{c}`" for c in tipo1},
                                 "_run_id": "s._run_id", "_ingested_at": "s._ingested_at",
                                 "_silver_updated_at": "s._silver_updated_at"})
    columnas = [c for c in spark.table(destino).columns if c in staged.columns]
    m.whenNotMatchedInsert("s._accion = 'INSERTAR'", {f"`{c}`": f"s.`{c}`" for c in columnas}).execute()


def escribir_merge_con_bajas(destino, df_ok, ausentes, pk):
    """MERGE (SCD1) que además marca bajas: _is_deleted y _deleted_at (cuándo lo detectó silver)."""
    src = (df_ok.withColumn("_deleted_at", F.when(F.col("_is_deleted"), F.col("_silver_updated_at")))
                .withColumn("_accion", F.lit("UPSERT")))
    if ausentes is not None:
        src = src.unionByName(ausentes.select(*[F.col(f"`{k}`") for k in pk],
                                              F.lit(True).alias("_is_deleted"),
                                              F.current_timestamp().alias("_deleted_at"),
                                              F.current_timestamp().alias("_silver_updated_at"),
                                              F.lit("BAJA").alias("_accion")),
                              allowMissingColumns=True)
    agregar_columnas_faltantes(destino, src)
    cond = " AND ".join(f"t.`{k}` <=> s.`{k}`" for k in pk)
    columnas = [c for c in src.columns if c != "_accion"]
    ins = {f"`{c}`": f"s.`{c}`" for c in columnas}
    upd = dict(ins)
    # una fila que sigue dada de baja conserva la fecha en que se detectó la baja
    upd["`_deleted_at`"] = "CASE WHEN coalesce(t._is_deleted, false) AND s._is_deleted THEN t._deleted_at ELSE s._deleted_at END"
    (DeltaTable.forName(spark, destino).alias("t").merge(src.alias("s"), cond)
        .whenMatchedUpdate("s._accion = 'BAJA' AND NOT coalesce(t._is_deleted, false)",
                           {"_is_deleted": "true", "_deleted_at": "s._deleted_at",
                            "_silver_updated_at": "s._silver_updated_at"})
        .whenMatchedUpdate("s._accion = 'UPSERT'", upd)
        .whenNotMatchedInsert("s._accion = 'UPSERT'", ins)
        .execute())

In [ ]:
try:
    origen_run = bronze_run_id or run_id
    if not spark.catalog.tableExists(e["bronze_table"]):
        resultado["status"] = "SKIPPED_NO_DATA"
    else:
        df = spark.table(e["bronze_table"]).where(F.col("_run_id") == origen_run)
        df = aplicar_columnas(df, e.get("columns")).persist()
        leidas = df.count()
        if leidas == 0:
            resultado.update(status="SKIPPED_NO_DATA", rows_read=0, rows_written=0)
        else:
            pii = [c["target_column"] for c in (e.get("columns") or []) if c.get("is_pii")]
            df_ok, rechazos, avisos = aplicar_calidad(df, e.get("dq_rules"), pii)

            if rechazos is not None:
                rechazos = rechazos.persist()
                n_rech = rechazos.count()
                if n_rech:
                    rechazos.write.format("delta").mode("append").saveAsTable(tabla_rechazos(e["silver_table"]))
                rechazos.unpersist()
            else:
                n_rech = 0

            pk = lista(e.get("primary_key"))
            df_ok = deduplicar(df_ok, pk, e.get("order_column"))
            df_ok = df_ok.withColumn("_silver_updated_at", F.current_timestamp())

            destino, modo = e["silver_table"], e["silver_mode"]
            bajas = (e.get("delete_mode") or "NONE").upper()
            existe = spark.catalog.tableExists(destino)
            historia = {}
            if bajas != "NONE":
                df_ok = marcar_borrado(df_ok, bajas, e.get("delete_flag_expr"))

            # Bajas por ausencia: la extracción es completa, lo que estaba vivo y no vino se borró en el origen.
            # Se comparan las claves leídas ANTES de calidad: una fila rechazada por DROP no es una baja.
            def ausentes_de(vivas):
                if bajas != "ABSENT" or not existe:
                    return None
                if bronze_run_id:
                    historia["bajas_no_evaluadas"] = "reproceso de una corrida vieja"
                    return None
                aus = claves_ausentes(vivas, df, pk).persist()
                n_vivas, n_aus = vivas.count(), aus.count()
                max_pct = e.get("delete_max_pct")
                controlar_bajas(n_vivas, n_aus, 10 if max_pct is None else max_pct)
                historia["bajas_por_ausencia"] = n_aus
                return aus

            if modo == "SCD2":
                seguidas = columnas_seguidas(df_ok, pk, lista(e.get("scd2_columns")), e.get("order_column"))
                if bajas == "NONE":
                    df_ok = marcar_borrado(df_ok, "NONE", None)
                forma = forma_version(df_ok, seguidas)
                if not existe:
                    forma.limit(0).write.format("delta").saveAsTable(destino)
                else:
                    if convertir_a_scd2(destino, seguidas):
                        historia["convertida_a_scd2"] = True
                    agregar_columnas_faltantes(destino, forma)
                tabla = spark.table(destino)
                vigentes = tabla.where("_is_current")
                staged, fuera = preparar_scd2(df_ok, vigentes, tabla, pk, seguidas, e.get("order_column"),
                                              ausentes_de(vigentes), pii)
                staged = staged.persist()
                cuenta = {r["_accion"]: r["n"] for r in staged.groupBy("_accion").agg(F.count("*").alias("n")).collect()}
                historia.update(versiones_nuevas=cuenta.get("INSERTAR", 0), versiones_cerradas=cuenta.get("CERRAR", 0),
                                bajas=staged.where("_accion = 'CERRAR' AND _is_deleted").count(),
                                fuera_de_orden=fuera)
                escribir_scd2(destino, staged, pk, seguidas, pii)
                staged.unpersist()
            elif modo == "MERGE" and bajas != "NONE" and existe:
                vivas = spark.table(destino)
                if "_is_deleted" in vivas.columns:
                    vivas = vivas.where(~F.coalesce(F.col("_is_deleted"), F.lit(False)))
                historia["filas_marcadas_baja"] = df_ok.where("_is_deleted").count()
                escribir_merge_con_bajas(destino, df_ok, ausentes_de(vivas), pk)
            elif modo == "OVERWRITE" or not existe:
                if bajas != "NONE":
                    df_ok = df_ok.withColumn("_deleted_at", F.when(F.col("_is_deleted"), F.col("_silver_updated_at")))
                (df_ok.write.format("delta").mode("overwrite")
                      .option("overwriteSchema", "true").saveAsTable(destino))
            elif modo == "APPEND":
                df_ok.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(destino)
            else:  # MERGE: upsert por clave
                cond = " AND ".join(f"t.`{k}` <=> s.`{k}`" for k in pk)
                spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
                (DeltaTable.forName(spark, destino).alias("t")
                    .merge(df_ok.alias("s"), cond)
                    .whenMatchedUpdateAll()
                    .whenNotMatchedInsertAll()
                    .execute())
            escritas = filas_escritas(destino)   # antes del ALTER, que también queda en el historial

            if pii:
                # Deja marcadas las columnas PII en la tabla, para gobierno y para la skill de documentación
                spark.sql(f"ALTER TABLE {destino} SET TBLPROPERTIES ('mdp.pii_columns' = '{','.join(pii)}')")

            resultado.update(status="SUCCEEDED", rows_read=leidas, rows_written=escritas,
                             rows_rejected=n_rech, dq_warnings=avisos or None, historia=historia or None)
        df.unpersist()
except Exception as ex:
    resultado.update(status="FAILED", error=f"{type(ex).__name__}: {ex}"[:3000])

print(json.dumps(resultado, ensure_ascii=False, indent=2, default=str))

In [ ]:
notebookutils.notebook.exit(json.dumps(resultado, ensure_ascii=False, default=str))